In [1]:
import pandas as pd

In [2]:
from pathlib import Path
data_path = Path("data.xls")

xls = pd.ExcelFile(data_path)
sheets = xls.sheet_names
print(sheets)

['日资料', '时段资料', '子流域', '场次', '参数']


In [3]:
daily_data_raw = pd.read_excel(data_path, sheet_name=sheets[0])
hourly_data_raw = pd.read_excel(data_path, sheet_name=sheets[1])

daily_data_raw.head()
hourly_data_raw.head()

,时间,实测流量,呈村,汪村,樟源口,棣甸,董坑坞,用功城,左龙,冯村,田里,大连
0,1989041102,6.95,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0
1,1989041103,7.20,0.0,1.3,1.3,0.942857,1.3,0.942857,0.0,0.942857,1.5,1.2
2,1989041104,7.45,1.5,1.3,1.3,1.342857,1.3,1.342857,1.3,1.342857,1.5,1.2
3,1989041105,7.70,1.5,1.3,1.3,1.342857,1.3,1.342857,1.3,1.342857,1.5,1.2
4,1989041106,7.95,1.4,1.3,1.3,1.328571,1.3,1.328571,1.3,1.328571,1.5,1.2


In [4]:
# rename columns for consistency
# 1. 时间 -> date/datetime
daily_data_raw["date"] = pd.to_datetime(
    daily_data_raw["时间"].astype(str),
    format="%Y%m%d"
)
hourly_data_raw["datetime"] = pd.to_datetime(
    hourly_data_raw["时间"].astype(str),
    format="%Y%m%d%H"
)
# 2.列名规整：中文站名 -> 英文 slug (rain_<station>)
STATION_SLUG = {
    "呈村":   "chengcun",
    "汪村":   "wangcun",
    "樟源口": "zhangyuankou",
    "棣甸":   "didian",
    "董坑坞": "dongkengwu",
    "用功城": "yonggongcheng",
    "左龙":   "zuolong",
    "冯村":   "fengcun",
    "田里":   "tianli",
    "大连":   "dalian",
}

def rename_ts_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    new_cols = {}
    for c in df.columns:
        if c == "实测流量":
            new_cols[c] = "q_obs"
        elif c == "蒸发":
            new_cols[c] = "evap"
        elif c in STATION_SLUG:
            new_cols[c] = f"rain_{STATION_SLUG[c]}"
    return df.rename(columns=new_cols)

daily_data = rename_ts_columns(daily_data_raw)
hourly_data = rename_ts_columns(hourly_data_raw)

print("[daily]  columns:", daily_data.columns.tolist())
print("[hourly] columns:", hourly_data.columns.tolist())

[daily]  columns: ['时间', 'q_obs', 'evap', 'rain_chengcun', 'rain_wangcun', 'rain_zhangyuankou', 'rain_didian', 'rain_dongkengwu', 'rain_yonggongcheng', 'rain_zuolong', 'rain_fengcun', 'rain_tianli', 'rain_dalian', 'date']
[hourly] columns: ['时间', 'q_obs', 'rain_chengcun', 'rain_wangcun', 'rain_zhangyuankou', 'rain_didian', 'rain_dongkengwu', 'rain_yonggongcheng', 'rain_zuolong', 'rain_fengcun', 'rain_tianli', 'rain_dalian', 'datetime']


In [5]:
# ===== 场次 / 子流域 / 参数 sheet 读取与拆分 =====

import yaml
subbasins_raw = pd.read_excel(data_path, sheet_name="子流域")
events_raw    = pd.read_excel(data_path, sheet_name="场次")
params_raw    = pd.read_excel(data_path, sheet_name="参数")

# 2) 子流域表规整
subbasins = subbasins_raw.copy()
subbasins.columns = ["idx", "station_cn", "area_km2", "area_ratio",
                     "muskingum_n_daily", "muskingum_n_event"]
subbasins["station"] = subbasins["station_cn"].map(STATION_SLUG)
subbasins["idx"] = subbasins["idx"].astype(int)

# 3) 加 meta 字段
ratio_sum = round(float(subbasins["area_ratio"].sum()), 6)
area_sum  = round(float(subbasins["area_km2"].sum()), 4)
subbasins_payload = {
    "meta": {
        "area_ratio_sum": ratio_sum,
        "area_km2_sum":   area_sum,
        "is_consistent":  abs(ratio_sum - 1.0) < 0.005,
    },
    "subbasins": subbasins.to_dict(orient="records"),
}
with open("subbasins.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(subbasins_payload, f, allow_unicode=True, sort_keys=False)
print(f"✅ subbasins.yaml  ({len(subbasins)} 行, area_ratio.sum = {ratio_sum})")

# 4) 场次表拆分（第 0 行是表头,真正数据从第 1 行起）
events_daily = events_raw.iloc[1:, 0:7].copy()
events_daily.columns = ["idx", "t_start", "t_end", "purpose", "p1", "p2", "p3"]
events_daily = (events_daily.dropna(subset=["idx"])
                            .drop(columns=["p1", "p2", "p3"], errors="ignore")
                            .reset_index(drop=True))
events_daily["idx"] = events_daily["idx"].astype(int)
events_daily["t_start"] = pd.to_datetime(events_daily["t_start"]).dt.strftime("%Y-%m-%d")
events_daily["t_end"]   = pd.to_datetime(events_daily["t_end"]).dt.strftime("%Y-%m-%d")
events_daily["purpose"] = events_daily["purpose"].fillna("")

events_event = events_raw.iloc[1:, 7:12].copy()
events_event.columns = ["idx", "t_start", "t_end", "purpose", "p1"]
events_event = (events_event.dropna(subset=["idx"])
                            .drop(columns=["p1"], errors="ignore")
                            .reset_index(drop=True))
events_event["idx"] = events_event["idx"].astype(int)
events_event["t_start"] = pd.to_datetime(events_event["t_start"]).dt.strftime("%Y-%m-%d %H:%M")
events_event["t_end"]   = pd.to_datetime(events_event["t_end"]).dt.strftime("%Y-%m-%d %H:%M")
events_event["purpose"] = events_event["purpose"].fillna("验证")

with open("events_daily.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump({"events_daily": events_daily.to_dict(orient="records")},
                   f, allow_unicode=True, sort_keys=False)
with open("events_event.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump({"events_event": events_event.to_dict(orient="records")},
                   f, allow_unicode=True, sort_keys=False)
print(f"✅ events_daily.yaml ({len(events_daily)} 场)")
print(f"✅ events_event.yaml ({len(events_event)} 场)")

# 5) 参数参考表
params_ref = params_raw.copy()
params_ref.columns = ["name", "meaning", "ref_value"]
params_ref["name"] = params_ref["name"].str.rstrip("：: ").str.strip()
params_ref = params_ref.where(pd.notnull(params_ref), None)
with open("params_ref.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump({"params": params_ref.to_dict(orient="records")},
                   f, allow_unicode=True, sort_keys=False)
print(f"✅ params_ref.yaml  ({len(params_ref)} 行)")


✅ subbasins.yaml  (10 行, area_ratio.sum = 1.0)
✅ events_daily.yaml (8 场)
✅ events_event.yaml (15 场)
✅ params_ref.yaml  (16 行)


In [6]:
# 列出两份数据的时间跨度以及检验是否完整
print("Daily data time span:", daily_data_raw["date"].min(), "to", daily_data_raw["date"].max())
print("Hourly data time span:", hourly_data_raw["datetime"].min(), "to", hourly_data_raw["datetime"].max())
print("Daily data missing values:\n", daily_data_raw.isnull().sum())
print("Hourly data missing values:\n", hourly_data_raw.isnull().sum())

Daily data time span: 1989-01-01 00:00:00 to 1996-12-31 00:00:00
Hourly data time span: 1989-04-11 02:00:00 to 1996-07-04 20:00:00
Daily data missing values:
 时间      0
实测流量    0
蒸发      0
呈村      0
汪村      0
樟源口     0
棣甸      0
董坑坞     0
用功城     0
左龙      0
冯村      0
田里      0
大连      0
date    0
dtype: int64
Hourly data missing values:
 时间          0
实测流量        0
呈村          0
汪村          0
樟源口         0
棣甸          0
董坑坞         0
用功城         0
左龙          0
冯村          0
田里          0
大连          0
datetime    0
dtype: int64


In [8]:
# ===== 落盘: 新安江模型可用的单文件 csv =====
# xaj_daily.csv 包含 1) 10 个雨量站降雨  2) 蒸发  3) 实测流量  4) imputed_flag
# 下游用法:
#   df = pd.read_csv("xaj_daily.csv", parse_dates=["time"], index_col="time")
#   rain_cols = [c for c in df.columns if c.startswith("rain_")]
#   rain = df[rain_cols]            # 10 列降雨 -> 面积加权
#   evap = df["evap"]               # 蒸发
#   q_obs = df["q_obs"]             # 实测流量 -> 率定目标
#   df_clean = df[~df["imputed_flag"]]   # 剔除隐式缺失行(可选)

out = daily_data.copy()
out.to_csv("xaj_daily.csv", index_label="time", encoding="utf-8-sig")
print(f"   columns: {out.columns.tolist()}")


   columns: ['时间', 'q_obs', 'evap', 'rain_chengcun', 'rain_wangcun', 'rain_zhangyuankou', 'rain_didian', 'rain_dongkengwu', 'rain_yonggongcheng', 'rain_zuolong', 'rain_fengcun', 'rain_tianli', 'rain_dalian', 'date']


In [9]:
# ===== 落盘: xaj_hourly.csv =====
# 流程: 时段资料 sheet  →  时间列转 YYYY-MM-DD HH:MM  →  落盘带 time 字符串列
# 用途: 次洪模型 (1h) 直接读这个文件

import pandas as pd
from pathlib import Path

xls = pd.ExcelFile("data.xls")
df = pd.read_excel(xls, sheet_name="时段资料")

# 1) 时间列: int 1989041102 → "1989-04-11 02:00"
df["time"] = pd.to_datetime(df["时间"].astype(str), format="%Y%m%d%H").dt.strftime("%Y-%m-%d %H:%M")

# 2) 删残留
df = df.drop(columns=["时间"])

# 3) 列重命名 (沿用 xaj_daily 同一套 STATION_SLUG 字典)
STATION_SLUG = {
    "呈村":   "chengcun",  "汪村":   "wangcun",
    "樟源口": "zhangyuankou", "棣甸":  "didian",
    "董坑坞": "dongkengwu",  "用功城": "yonggongcheng",
    "左龙":   "zuolong",   "冯村":  "fengcun",
    "田里":   "tianli",    "大连":  "dalian",
}
rename = {"实测流量": "q_obs"}
for cn, slug in STATION_SLUG.items():
    rename[cn] = f"rain_{slug}"
df = df.rename(columns=rename)

# 4) 重排 time 在前
cols = ["time"] + [c for c in df.columns if c != "time"]
df = df[cols]

# 5) 落盘
df.to_csv("xaj_hourly.csv", index=False, encoding="utf-8-sig")
print(f"✅ xaj_hourly.csv  shape={df.shape}")
print(f"   time: {df['time'].iloc[0]} ... {df['time'].iloc[-1]}")
print(f"   cols: {df.columns.tolist()[:5]} ...")



✅ xaj_hourly.csv  shape=(5683, 12)
   time: 1989-04-11 02:00 ... 1996-07-04 20:00
   cols: ['time', 'q_obs', 'rain_chengcun', 'rain_wangcun', 'rain_zhangyuankou'] ...
